In [57]:
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import io
from astropy.io import fits
import ipywidgets as widgets
from IPython.display import display
from sklearn.model_selection import KFold
from torch import nn
from torchvision import transforms
from torchvision.transforms import Normalize

In [58]:
cwd = Path.cwd()
home_dir = cwd.parent.parent
cutout_dir = home_dir / 'data' / 'cutouts'
utils_path = home_dir / 'utils.py'

In [59]:
sys.path.append(str(home_dir))
import utils as ut

In [60]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
SEED = 42
BEST_ACC = 0

# HYPERPARAMETERS
INPUT_SIZE = 67
EPOCHS = 200
NUM_TRIALS = 20
PATIENCE = 20
SCALER = 'tanh'
N_SPLITS = 5
AUG_MULT = 5
BANDS = ['f150w', 'f277w', 'f444w']
OPTIMIZE = False

In [61]:
if OPTIMIZE:
    COMMENT = input('Enter OPTIMIZATION comment: ')

In [62]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [63]:
# LOAD DATA INTO DF

cols = {'idx': [], 'images': [], 'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = {}
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)

df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
df = df.sample(frac=1, random_state=SEED) if SEED else df.sample(frac=1)

In [64]:
# STACK AND RESIZE BANDS

resize_transform = transforms.Resize(
    INPUT_SIZE, transforms.InterpolationMode.BILINEAR
)

df['images'] = df[BANDS].values.tolist()
df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]

In [65]:
# EXTRACT X, y, and i from data

data_df = df[df['TABLE'] != 'test']

X_data = torch.stack(data_df['images'].tolist())
y_data = torch.tensor(data_df['FLAG'].tolist())
i_data = torch.tensor(data_df['idx'].astype(int).tolist())

X_data.shape

torch.Size([438, 3, 67, 67])

In [66]:
# CLIP NEGATIVE VALUES

X_data = torch.clamp(X_data, min=0)

In [67]:
# SCALE DATA


def scale_data(data, type='tanh'):

    if type == 'log1p':
        data = torch.log1p(data)
    elif type == 'pytorch_normalize':
        scaler = Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
        data = scaler(data)
    elif type == 'arcsinh':
        data = torch.asinh(data)
    elif type == 'tanh':
        data = torch.tanh(data)
    else:
        raise ValueError(f'{type} is not a valid scaling function')

    return data


X_data_scaled = scale_data(X_data, type=SCALER)

ut.analyze_data(X_data, axis=(0, 2, 3), name='Unscaled data:')
print('\n')
ut.analyze_data(X_data_scaled, axis=(0, 2, 3), name=f'{SCALER} Scaled data:')

Unscaled data:
Mean:  [0.04344692 0.10138565 0.17207925]
Median:  [0.01113511 0.01979694 0.02893846]
St. Dev:  [0.17097937 0.38211322 0.7541184 ]
Max:  [17.483625 34.802776 80.58712 ]
Min:  [0. 0. 0.]
Length:  438


tanh Scaled data:
Mean:  [0.03840174 0.07874494 0.10869451]
Median:  [0.01113465 0.01979436 0.02893038]
St. Dev:  [0.09151433 0.15734759 0.20512508]
Max:  [1. 1. 1.]
Min:  [0. 0. 0.]
Length:  438


In [68]:
# CONVERT TO DATSET

# transforms to apply to inputs
transform = transforms.Compose(
    [
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=(0, 360))
        # transforms.RandomPerspective()
    ]
)

dataset = ut.GalaxyDataset(
    images=X_data_scaled,
    labels=y_data,
    transform=transform,
    multiplier=AUG_MULT,
)

dataset.images[0].shape

torch.Size([3, 67, 67])

In [76]:
def show_galaxy(index):
    img = dataset.images[index]
    print(f'Selected Index: {index}')
    return ut.imshow_tensor(img)


index_dropdown = widgets.Dropdown(
    options=range(len(dataset)),
    description='Index:',
)

widgets.interact(show_galaxy, index=index_dropdown)

interactive(children=(Dropdown(description='Index:', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14…

<function __main__.show_galaxy(index)>

## Define Model

In [70]:
# DEFINE MODEL FUNCTION


def define_model(trial):
    # get model structure from optuna
    n_conv = trial.suggest_int('n_conv', 1, 3)
    n_fc = trial.suggest_int('n_fc', 1, 3)

    conv_channels = [
        trial.suggest_int(f'n_channels_conv{i}', 1, 32) for i in range(3)
    ]
    conv_ksizes = [
        trial.suggest_int(f'kernel_size_conv{i}', 1, 3) for i in range(3)
    ]
    conv_paddings = [
        trial.suggest_int(f'padding_conv{i}', 1, 3) for i in range(3)
    ]
    conv_dropouts = [
        trial.suggest_float(f'dropout_conv{i}', 0.2, 0.5) for i in range(3)
    ]
    fc_units = [
        trial.suggest_int(f'n_units_fc{i}', 4, 256, step=4) for i in range(3)
    ]
    fc_dropouts = [
        trial.suggest_float(f'dropout_fc{i}', 0.2, 0.5) for i in range(3)
    ]

    layers = []

    # build conv layers
    in_channels = len(BANDS)
    current_spatial_size = INPUT_SIZE

    for i in range(n_conv):
        out_channels = conv_channels[i]
        ksize = conv_ksizes[i]
        pad = conv_paddings[i]
        dropout = conv_dropouts[i]

        layers.extend(
            (
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=ksize,
                    padding=pad,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        current_spatial_size = current_spatial_size - ksize + (2 * pad) + 1
        in_channels = out_channels

    flattened_features = in_channels * (current_spatial_size**2)
    layers.append(nn.Flatten())

    # build fc layers
    in_features = flattened_features
    for i in range(n_fc):
        out_features = fc_units[i]
        dropout = fc_dropouts[i]

        layers.extend(
            (
                nn.Linear(in_features, out_features, bias=False),
                nn.BatchNorm1d(out_features),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        in_features = out_features

    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)

In [71]:
# DEFINE OBJECTIVE FUNCTION


def objective(trial):
    global BEST_ACC

    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accuracies = []

    # Split the dataset into 5 folds
    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        # create new model
        model = define_model(trial).to(DEVICE)
        optimizer, loss_func, batch_size = ut.get_hyperparameters(trial, model)

        # get separate val and train features and labels and scale
        features, labels = dataset.images, dataset.labels
        X_train, X_val = features[train_idx], features[val_idx]
        y_train, y_val = labels[train_idx], labels[val_idx]

        # create subsets and dataloaders
        train_subset = ut.GalaxyDataset(
            images=X_train,
            labels=y_train,
        )
        val_subset = ut.GalaxyDataset(images=X_val, labels=y_val)

        if SEED:
            train_loader = torch.utils.data.DataLoader(
                train_subset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=True,
            )
        else:
            train_loader = torch.utils.data.DataLoader(
                train_subset,
                batch_size=batch_size,
                shuffle=True,
                drop_last=True,
            )
        val_loader = torch.utils.data.DataLoader(
            val_subset, batch_size=batch_size, shuffle=False
        )

        # early stopper
        stopper = ut.EarlyStopper(patience=PATIENCE, min_delta=0.0005)
        for _ in range(EPOCHS):
            # Training Loop ----------------------------------------------------
            model.train()
            ut.do_epoch(
                model=model,
                loader=train_loader,
                loss_function=loss_func,
                device=DEVICE,
                optimizer=optimizer,
                train=True,
            )

            # Validation Loop ---------------------------------------------------
            model.eval()
            with torch.no_grad():
                running_loss, correct = ut.do_epoch(
                    model=model,
                    loader=val_loader,
                    loss_function=loss_func,
                    device=DEVICE,
                    train=False,
                )

                # compute accuracy
                accuracy = correct / len(val_subset)
                if accuracy > BEST_ACC:
                    BEST_ACC = float(accuracy)

                # track loss once per epoch to check early stopper
                val_loss = running_loss / len(val_subset)
                stopper(val_loss)
                if stopper.early_stop:
                    break

        fold_accuracies.append(float(accuracy))
        trial.report(accuracy, step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_accuracies)

## Train Model

In [72]:
# TRAINS MODEL

if OPTIMIZE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # use gaussian process
    sampler = (
        optuna.samplers.GPSampler(seed=SEED)
        if SEED
        else optuna.samplers.GPSampler()
    )

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(),
    )
    study.optimize(objective, n_trials=NUM_TRIALS, show_progress_bar=True)

    print('Best trial:')
    trial = study.best_trial
    print(f'  Accuracy: {trial.value}')
    print('  Params: ')
    for key, value in trial.params.items():
        print(f'    {key}: {value}')

In [73]:
# LOG OPTIMIZATION

if OPTIMIZE:
    logged_params = trial.params.copy()
    logged_params['comment'] = COMMENT
    logged_params['bands'] = BANDS
    logged_params['out_features'] = OUT_FEATURES
    logged_params['seed'] = SEED
    logged_params['epochs'] = EPOCHS
    logged_params['image_size'] = INPUT_SIZE
    logged_params['num_trials'] = NUM_TRIALS
    logged_params['patience'] = PATIENCE
    logged_params['accuracy'] = trial.value
    logged_params['best accuracy'] = BEST_ACC
    logged_params['scaling'] = SCALER
    logged_params['N_folds'] = N_SPLITS
    logged_params['aug_mult'] = AUG_MULT

    logger = ut.RunLogger(filepath=home_dir / 'logs' / 'conv.json')
    logger.log_run(logged_params)